# Setup

In [ ]:
import os

default_n_threads = 32
os.environ["OPENBLAS_NUM_THREADS"] = f"{default_n_threads}"
os.environ["MKL_NUM_THREADS"] = f"{default_n_threads}"
os.environ["OMP_NUM_THREADS"] = f"{default_n_threads}"
os.environ["NUM_THREADS"] = f"{default_n_threads}"
import sys
import pickle
import numpy as np

sys.path.append(os.path.abspath("../../"))
import gc
import importlib
import warnings
import correlation
import distance
import kmeans
import lvc
import density as Density
import matplotlib.pyplot as plt
import matplotlib as mpl
import plotting
import read
import utils
from tqdm import tqdm
import scienceplots
import matplotlib.lines as mlines
from matplotlib.ticker import FuncFormatter
import string

warnings.filterwarnings("ignore", category=FutureWarning)
import plotly.io as pio

pio.renderers.default = "notebook"


def reload_utils():
    importlib.reload(utils)
    importlib.reload(kmeans)
    importlib.reload(correlation)
    importlib.reload(plotting)
    importlib.reload(read)
    importlib.reload(distance)
    importlib.reload(lvc)
    importlib.reload(Density)
    importlib.reload(scienceplots)
    plt.style.use("science")
    plt.rcParams["font.sans-serif"] = ["Arial"]
    plt.rcParams["font.family"] = "sans-serif"
    mpl.rcParams.update(
        {
            "text.usetex": True,
            "text.latex.preamble": r"""
            \usepackage[scaled]{helvet}    % Nimbus Sans L ≈ Arial
            \usepackage[EULERGREEK]{sansmath}  % math in sans serif
            \usepackage{bm}
            \usepackage{amsmath}
            \usepackage{amsfonts}
            \DeclareMathSymbol{\shortminus}{\mathbin}{AMSa}{"39}
            \sansmath
            \renewcommand{\familydefault}{\sfdefault}   % default to sans
        """,
        }
    )
    np.set_printoptions(suppress=True)
    return None


reload_utils()

######### setup #########

mol = "silanol_trimethyl"
kahuna = True
cluster = False
AMU_TO_KG = 1.66054e-27

mol_map = {
    "cyclotrisiloxane": "mol1",
    "silanol_trimethyl": "mol2",
    "trisiloxane": "mol3",
}

paper_names = {
    "trisiloxane": "TSH",
    "cyclotrisiloxane": "CTS",
    "silanol_trimethyl": "ST",
}

mol_colors = {
    "cyclotrisiloxane": "black",
    "silanol_trimethyl": "goldenrod",
    "trisiloxane": "darkred",
}
mol_markers = {"cyclotrisiloxane": "o", "silanol_trimethyl": "s", "trisiloxane": "^"}
mol_color = mol_colors[mol]
mol_marker = mol_markers[mol]


paper_name = paper_names[mol]

if kahuna:
    results_dir = f"/home/gtdebru/phase/{mol}/lvc/results/"
    data_dir = f"/projects/wg-gas-surface/LVC/{mol_map[mol]}/"
    atomic_masses_path = f"/home/gtdebru/phase/{mol}/lvc/atomic_masses.pkl"
    fig_path = f"/home/gtdebru/phase/{mol}/lvc/figures/"

elif cluster:
    results_dir = f"/home/gtdebru/phase/{mol}/lvc/results/"
    data_dir = f"/projects/AL_gas_surface/LVC/{mol_map[mol]}/"
    atomic_masses_path = f"/home/gtdebru/phase/{mol}/lvc/atomic_masses.pkl"
    fig_path = f"/home/gtdebru/phase/{mol}/lvc/figures/"

else:
    results_dir = f"/Users/gtdebru/phase/{mol}/lvc/results/"
    data_dir = f"/Users/gtdebru/phase/{mol}/lvc/data/"
    atomic_masses_path = f"/Users/gtdebru/phase/{mol}/lvc/atomic_masses.pkl"
    fig_path = f"/Users/gtdebru/phase/{mol}/lvc/figures/"
    temps = [temp for temp in range(200, 625, 50)]

with open(atomic_masses_path, "rb") as f:
    atomic_masses = pickle.load(f)

temps = [t for t in range(200, 625, 25)]

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

if not os.path.exists(results_dir):
    os.makedirs(results_dir)

if not os.path.exists(data_dir + "molecule_dumps"):
    os.makedirs(data_dir + "molecule_dumps")

if not os.path.exists(fig_path + "transistion-rates"):
    os.makedirs(fig_path + "transistion-rates")

if not os.path.exists(fig_path + "kmeans-density-fits"):
    os.makedirs(fig_path + "kmeans-density-fits")

BETA = 0.325  # order-parameter exponent
ALPHA = 0.11  # heat-capacity exponent (for diameter correction)

In [ ]:
#### Check Plotting Tools ####
import shutil

for tool in ("latex", "dvipng", "dvisvgm", "gs", "kpsewhich"):
    print(f"{tool:8s} →", shutil.which(tool))

print("Matplotlib cache:", mpl.get_cachedir())

# Feature Selection

#### The last coexistence temperature is 400K

In [ ]:
### Setup Feature Sweep and Temperature Range ###

reload_utils()
lags = np.array([1, 2, 3, 4])
radii = np.arange(6, 25)
ntemps = len(temps)
nvars = 3
nt = 1001
end = temps.index(400) + 1
temps = np.array(temps)
coexist_temps = temps[:end]
ncoexist = len(coexist_temps)
train_temp = 400

print(coexist_temps)

In [ ]:
print(temps[end:])

In [ ]:
### Read Train Data ###

data_path = f"{data_dir}LVC_{mol}_eq{train_temp}K_a.dump"
df_train = read.process_data(
    data_path, atomic_masses, metal_type=-1, nt_lim=None, to_print=True
)
utils.generate_displacement(df_train, lags, to_print=True)
Density.atomic_density(df_train, radii, to_print=True)
df_train["paper_name"] = paper_name

gyr = utils.rad_gyration(df_train)
max_r = utils.molecule_size(df_train)
print(f"Radius of Gyration: {gyr:.2f}Å", f"Max Size: {max_r:.2f}Å")

In [ ]:
### Perform Hyperparameter Grid Search ###

errs = kmeans.feat_optim(df_train, radii, lags)

In [ ]:
### Plot Results ###

sil = errs["sil"][::-1, :]  # reverse for plotting
r = errs["rmse"][::-1, :]
fig, ax = plotting.dual_heatmap(
    sil, r, radii, lags, cmap="rocket", annot_font=10, fmt=".3f"
)
fig.savefig(fig_path + f"hyperparameter_{paper_name}.png")
plt.show()

In [ ]:
### Select Optimized Radius ###
argmin = np.argmin(errs["rmse"][1:, 0]) + 1  # Select rmse minimizer after the first row
rad = radii[argmin]
print("Selected Radius:", rad, "Å")

# Training

In [ ]:
### Train Data Classification, Extract Centroids & Minmax ###

reload_utils()

lag = 1
lags = [lag]
radii = [rad]
cluster_vars = [f"density_{rad}", f"displacement_{lag}", "pe"]

print("Cluster Variables:\n", cluster_vars)

molecule_phase, atom_phase, centroids, minmax = kmeans.classify_phase(
    df_train,
    cluster_vars,
    norm="minmax",
    return_minmax=True,
)

raw_centroids = df_train["raw_centroids"].copy()
start = 1

np.savetxt(results_dir + "normalized_centroids.txt", centroids)
np.savetxt(results_dir + "raw_centroids.txt", raw_centroids)
np.savetxt(results_dir + "cluster_vars.txt", np.array(cluster_vars, str), fmt="%s")

In [ ]:
print("Normalized Centroids:\n")
arrays = {"var": cluster_vars, "mu_l": centroids[0], "mu_v": centroids[1]}
utils.print_rows(arrays)
print()

print("Raw Centroids:\n")
arrays = {"var": cluster_vars, "mu_l": raw_centroids[0], "mu_v": raw_centroids[1]}
utils.print_rows(arrays)

In [ ]:
### Read in Coexistance Data ###

reload_utils()
dfs = {}
for t in tqdm(range(ncoexist)):

    temp = coexist_temps[t]

    if temp == train_temp:
        dfs[temp] = df_train
        continue

    data_path = f"{data_dir}LVC_{mol}_eq{temp}K_a.dump"
    df_temp = read.process_data(
        data_path, atomic_masses, metal_type=-1, nt_lim=None, to_print=False
    )
    utils.generate_displacement(df_temp, lags)
    Density.atomic_density(df_temp, radii)
    df_temp["paper_name"] = paper_name
    dfs[temp] = df_temp

In [ ]:
utils.print_memory()

# Classify & Extract Density Profiles

In [ ]:
### Fit the LVC / KMeans Densities and Feature Z-Profiles ###

reload_utils()
bin_width = 4
dz = 4
lvc_densities = []
feat_zs = []
data_densities = []
kmeans_densities = []
kmeans_density_fits = []
phases = []
features = []  # so we can look at the whole feature distribution

for t in tqdm(range(ncoexist)):

    temp = coexist_temps[t]
    df_temp = dfs[temp]

    ######## LVC #######
    lvc_density = lvc.fit_density(
        df_temp, bin_width=bin_width, mode="atom", auto_range=False
    )
    # normed feats v z
    feat_zbin, feat_val = utils.feats_z(df_temp, cluster_vars, dz=dz, norm=minmax)
    # data density
    data_density, data_zbin, err_data = Density.density(
        df_temp,
        bin_width=bin_width,
        time_avg=True,
        mode="atom",
        std=True,
        absval=True,
        center=True,
        auto_range=True,
        norm="mass",
        actime=False,
        start=0,
    )

    lvc_densities.append(lvc_density)
    feat_zs.append([feat_zbin, feat_val])
    data_densities.append([data_zbin, data_density])

    ######## K-Means #######

    _ = kmeans.classify_phase(
        df_temp,
        cluster_vars,
        norm=minmax,  # minmax rel. train features
        external_centroids=centroids,
    )

    kmeans_density = kmeans.density(df_temp, bin_width=bin_width)
    kmeans_density_fit = kmeans.fit_density(kmeans_density, temp)
    kmeans_densities.append(kmeans_density)
    kmeans_density_fits.append(kmeans_density_fit)
    phases.append(utils.parse(df_temp, "phase")[lag:])
    feats = [utils.parse(df_temp, var)[lag:] for var in cluster_vars]
    features.append(feats)  # Save features for plots

    data_path = f"{data_dir}LVC_{mol}_eq{temp}K_a.dump"

    utils.write_dump(
        df_temp,
        f"{data_dir}molecule_dumps/LVC_{mol}_eq{temp}K_a_molecule.dump",
        mode="molecule",
    )
    utils.write_phase(data_path, df_temp)
    utils.write_switch_info(
        df_temp,
        f"{results_dir}LVC_{mol}_eq{temp}K_a_rates.txt",
        centroids=centroids,
    )

    tscatter = 500
    title = f"{paper_name} {utils.get_t_ns(df_temp, tscatter):.2f} ns"
    fig = plotting.scatter_var(
        df_temp, "phase", t=tscatter, mode="molecule", center=True, title=title
    )
    path = fig_path + f"{paper_name}_{temp}K_scatter.png"
    plotting.write_plotly(fig, path, scale=2)

In [ ]:
### Get mins and maxes from across all the temps

global_mins = np.ones(nvars) * 1e9
global_maxes = np.ones(nvars) * -1e9

for temp_feats in features:

    for i in range(nvars):

        mi = temp_feats[i].min()
        ma = temp_feats[i].max()

        if mi < global_mins[i]:
            global_mins[i] = mi
        if ma > global_maxes[i]:
            global_maxes[i] = ma

In [ ]:
### Sanity Check Kmeans Density Values rho_g, rho_l ###

for temp, kmeans_dens in zip(coexist_temps, kmeans_densities):

    fig, ax = kmeans.plot_density_fit(kmeans_dens, temp)
    fig.savefig(fig_path + "kmeans-density-fits/" + f"kmeans-density-fit-{temp}K.png")
    plt.show()

# Transistion Probabilities & Trajectories

In [ ]:
# plot transistion rates and probabilities

for t, temp in enumerate(coexist_temps):
    df_temp = dfs[temp]
    fig, axs = plotting.plot_transistion_rate_prob(df_temp)
    path = fig_path + "transistion-rates/" + f"{paper_name}_{temp}K.png"
    fig.savefig(path)
    plt.show()

In [ ]:
# plot transistion locations

for t, temp in enumerate(coexist_temps):
    df_temp = dfs[temp]
    fig, ax = plotting.plot_transistions(df_temp, lvc_densities[t], rad)
    path = fig_path + f"{paper_name}_{temp}K_transistion_locations.png"
    fig.savefig(path)
    plt.show()

In [ ]:
# trajectory

temp = 300
ti = np.where(coexist_temps == temp)[0][0]
fig, ax = plotting.plot_molecule_trajectory(dfs[temp], lvc_densities[ti])
plt.show()

# Vapor Dome

In [ ]:
#### Check fit validity ###

rho_ls = np.array(kmeans_density_fits)[:, 0]
rho_gs = np.array(kmeans_density_fits)[:, 1]
crit = lvc.fit_critical(rho_ls, rho_gs, coexist_temps, corrected=False, p0=None)

Tc = crit["Tc"]
rhoc = crit["rhoc"] / 1000
print(f"Full range: Tc = {Tc:.2f}, rhoc = {rhoc:.4f}")

fig, ax = plt.subplots(ncols=2, dpi=400, figsize=(5, 2))

# mask = (mask) & (temps > 250)
x1 = coexist_temps
y1 = (rho_ls + rho_gs) / 2
m, b = np.polyfit(x1, y1, deg=1)


ax[0].plot(x1, y1, color="black")
ax[0].set_xlabel(r"$T \text{(K)}$")
ax[0].set_ylabel(r"$(\rho_{l} + \rho_{v}) / 2$")
ax[0].grid()

x2 = np.log10(Tc - coexist_temps)
y2 = np.log10(rho_ls - rho_gs)

m, b = np.polyfit(x2, y2, deg=1)
# print(m, b, 10**b)

ax[1].plot(x2, y2, color="black")
ax[1].set_xlabel(r"$\log (T_{c} - T)$")
ax[1].set_ylabel(r"$\log (\rho_{l} - \rho_{v})$")
ax[1].grid()


#### remove problem temps ###

mask = coexist_temps > 250

crit = lvc.fit_critical(
    rho_ls[mask], rho_gs[mask], coexist_temps[mask], corrected=False, p0=None
)
Tc_red = crit["Tc"]
rhoc_red = crit["rhoc"] / 1000

print(f"Reduced range: Tc = {Tc_red:.2f}, rhoc = {rhoc_red:.4f}")

### Quadratic correction

crit = lvc.fit_critical(rho_ls, rho_gs, coexist_temps, corrected=True, p0=None)
Tc_quad = crit["Tc"]
rhoc_quad = crit["rhoc"] / 1000

print(f"Corrected: Tc = {Tc_quad:.2f}, rhoc = {rhoc_quad:.4f}")

# fig.suptitle4
#     rf"$T_{{c}} = {Tc:.1f}\text{{K}} \ \ \rho_{{c}} = {rhoc/1000:.3f} \text{{ g cm}}^{{-3}}$"
# )

fig.tight_layout()
fig.savefig(fig_path + f"fit_check_{paper_name}.png")
plt.show()

In [ ]:
####### KMeans vs LVC critical values #######

mask = coexist_temps >= 200
rho_ls = np.array(kmeans_density_fits)[:, 0]
rho_gs = np.array(kmeans_density_fits)[:, 1]
crit = lvc.fit_critical(rho_ls[mask], rho_gs[mask], coexist_temps[mask])
rhoc = crit["rhoc"] / 1000
Tc = crit["Tc"]
rho_ls /= 1000
rho_gs /= 1000
print("kmeans:")
utils.print_rows({"T": coexist_temps, "rho_g": rho_gs, "rho_l": rho_ls})
print(f"\t Tc: {Tc:.2f} \n \t rhoc: {rhoc:.2f}")


fig, ax = plt.subplots(dpi=400)
s = 20
lw = 0.7

ax.plot(rho_ls, coexist_temps, color="red", label="KMeans")
ax.plot(rho_gs, coexist_temps, color="red")
ax.scatter(rhoc, Tc, marker="x", color="red")


##### LVC #####

rho_ls = []
rho_gs = []
for t in range(ncoexist):

    rho_ls.append(lvc_densities[t][3])
    rho_gs.append(lvc_densities[t][4])

rho_ls = np.array(rho_ls)
rho_gs = np.array(rho_gs)
crit = lvc.fit_critical(rho_ls[mask], rho_gs[mask], coexist_temps[mask])
rhoc = crit["rhoc"] / 1000
Tc = crit["Tc"]
a = crit["A"]
b = crit["B"]
rho_ls /= 1000
rho_gs /= 1000
print("lvc:")
utils.print_rows({"T": coexist_temps, "rho_g": rho_gs, "rho_l": rho_ls})
print(f"\t Tc: {Tc:.2f} \n \t rhoc: {rhoc:.2f}")


ax.scatter(
    rho_ls,
    coexist_temps,
    color="black",
    label="LVC",
    marker="o",
    s=s,
    zorder=2,
    facecolors="none",
    linewidths=lw,
)
ax.scatter(
    rho_gs,
    coexist_temps,
    color="black",
    marker="o",
    s=s,
    zorder=2,
    facecolors="none",
    linewidths=lw,
)


ax.scatter(rhoc, Tc, color="black", s=s, linewidths=lw, facecolors="none")
ax.set_xlabel(r"$\rho \ (\text{ g cm}^{-3}) $")
ax.set_ylabel(r"$T \ (\text{K})$")
ax.grid()
ax.legend()
fig.savefig(fig_path + "lvc-kmeans-vapor-dome.png")
plt.show()

In [ ]:
##### KMeans With Fit #####

rho_ls = np.array(kmeans_density_fits)[:, 0]
rho_gs = np.array(kmeans_density_fits)[:, 1]
# mask = np.ones_like(rho_ls, bool)
mask = coexist_temps >= 200
crit = lvc.fit_critical(rho_ls[mask], rho_gs[mask], coexist_temps[mask])
rhoc = crit["rhoc"]
Tc = crit["Tc"]
a = crit["A"]
b = crit["B"]
T_fit = np.linspace(temps[0], Tc - 10, 100)
rho_l_fit, rho_g_fit = lvc.coexistence_densities(T_fit, Tc, rhoc, a, b, BETA)


rho_ls /= 1000
rho_gs /= 1000
rhoc /= 1000
rho_l_fit /= 1000
rho_g_fit /= 1000

plot_info = {}
plot_info["Tc"] = Tc
plot_info["rhoc"] = rhoc
plot_info["T_fit"] = T_fit
plot_info["T"] = coexist_temps
plot_info["rho_l_fit"] = rho_l_fit
plot_info["rho_g_fit"] = rho_g_fit
plot_info["rho_l"] = rho_ls
plot_info["rho_g"] = rho_gs

filename = f"{results_dir}vapor_dome_{paper_name}.pkl"
with open(filename, "wb") as file:
    pickle.dump(plot_info, file)


fig, ax = plt.subplots(dpi=400)
s = 20
lw = 0.5
s = 30
mol_label = paper_name
color = mol_color
marker = mol_marker

ax.plot(rho_l_fit, T_fit, color=color)
ax.plot(rho_g_fit, T_fit, color=color)
ax.scatter(rhoc, Tc, marker=marker, color=color, s=s, edgecolors="none")
ax.scatter(
    rho_ls,
    coexist_temps,
    color=color,
    label=mol_label,
    marker=marker,
    s=s,
    zorder=2,
    facecolors="white",
    linewidths=lw,
)
ax.scatter(
    rho_gs,
    coexist_temps,
    color=color,
    marker=marker,
    s=s,
    zorder=2,
    facecolors="white",
    linewidths=lw,
)

# ax.scatter(
#     rhoc_red, Tc_red, marker="x", color="green", label="Reduced", s=s - 10, alpha=0.5
# )
# ax.scatter(
#     rhoc_quad, Tc_quad, marker="o", color="blue", label="Corrected", s=s - 10, alpha=0.5
# )

ax.set_xlabel(r"$\rho \ (\text{ g cm}^{-3}) $")
ax.set_ylabel(r"$T \ (\text{K})$")
ax.set_xlabel(r"$\rho \ (\text{g/cm}^{3}) $")
ax.set_ylabel(r"$T \ (\text{K})$")
ax.legend(handlelength=1, loc=(0.1, 0.2))
ax.grid()
fig.savefig(fig_path + f"vapor_dome_{paper_name}.png")
plt.show()

# Vapor Pressure

In [ ]:
### Read in Vapor Data and Get rho_g for Vapor Pressure ###

bin_width = 4
vapor_density_fits = []

for temp in tqdm(temps[end:]):

    data_path = f"{data_dir}LVC_{mol}_eq{temp}K_a.dump"
    df_temp = read.process_data(
        data_path, atomic_masses, metal_type=-1, nt_lim=None, to_print=False
    )
    utils.generate_displacement(df_temp, lags)
    Density.atomic_density(df_temp, radii)

    ######## K-Means #######

    _ = kmeans.classify_phase(
        df_temp,
        cluster_vars,
        norm=minmax,  # minmax rel. train features
        external_centroids=centroids,
    )

    kmeans_density = kmeans.density(df_temp, bin_width=bin_width)
    kmeans_density_fit = kmeans.fit_density(kmeans_density, temp)
    vapor_density_fits.append(kmeans_density_fit)

    utils.write_dump(
        df_temp,
        f"{data_dir}molecule_dumps/LVC_{mol}_eq{temp}K_a_molecule.dump",
        mode="molecule",
    )
    utils.write_phase(data_path, df_temp)
    utils.write_switch_info(
        df_temp,
        f"{results_dir}LVC_{mol}_eq{temp}K_a_rates.txt",
        centroids=centroids,
    )

    del df_temp
    gc.collect()

In [ ]:
fig, ax = plt.subplots(dpi=400)
s = 30
mol_label = paper_name

color = mol_color
marker = mol_marker

R = 8.314  # J/ (K mol)
A = 6.022e23  # particles per mol
m = df_train["molecule"]["mass"] * df_train["AMU_TO_KG"] * A

rho_gs_coexist = np.array(kmeans_density_fits)[:, 1]
rho_gs_vapor = np.array(vapor_density_fits)[:, 1]
rho_gs = np.concatenate((rho_gs_coexist, rho_gs_vapor))

rho_ls_coexist = np.array(kmeans_density_fits)[:, 0]
rho_ls_vapor = np.array(vapor_density_fits)[:, 0]
rho_ls = np.concatenate((rho_ls_coexist, rho_gs_vapor))

mask = rho_gs > 1e-4
rho_gs_plot = rho_gs[mask]

T = temps[mask]
P = rho_gs_plot * R * T / m / 1e6  # kg / (m * s^2)

ax.scatter(
    T,
    P,
    label=mol_label,
    marker=marker,
    s=s,
    color=color,
    zorder=10,
    facecolors="white",
    linewidths=0.5,
)

fit_end = np.where(T == 425)[0][0] + 1
T_fit = T[:fit_end]
P_fit = P[:fit_end]
m, b = np.polyfit(T_fit, np.log10(P_fit), deg=1)
x = np.linspace(T.min(), T[fit_end - 1], 1000)  # T's to plot
y = m * x + b
y = 10**y  # P

atm = 0.101325  # MPa
Tb = x[np.argmin(np.abs(y - atm))]  # Boiling Point

ax.plot(x, y, color=color, linestyle="--")
ax.grid(which="major", linestyle="-")
ax.grid(which="minor", linestyle=":")
ax.legend(handlelength=1, loc=(0.1, 0.65))
ax.set_yscale("log")
ax.set_xlabel(r"$T \ (\text{K})$")
ax.set_ylabel(r"$\text{Vapor Pressure (MPa)}$")
ax.set_title(rf"$T_{{b}} = {Tb:.2f}$K")
ax.legend()
fig.savefig(fig_path + f"vapor_pressure_{paper_name}.png")
plt.show()

vapor_pressure = {}

vapor_pressure["T"] = T
vapor_pressure["P"] = P
vapor_pressure["rho_g"] = rho_gs_plot
vapor_pressure["mass"] = m

filename = f"{results_dir}vapor_pressure_{paper_name}.pkl"
with open(filename, "wb") as file:
    pickle.dump(vapor_pressure, file)

# Z-profiles and Feature Distributions

In [ ]:
# The binned z profiles will have smaller extrema due to binning
# The binned z profiles will vary between a lower and higher fraction of the train normalization
# compared to the binned z profiles, the distribution bounds will be higher due to inclusion of individual extremes
# If we adjust the distribution limits to be the same normalized bounds as the z-profiles, the
# z-profiles should match the distributions
# This means that instead of +- 5% of the max unnormalized features, we should calculate the the unnormalized values that give rise
# to the max/min of the normalized features

# The rho_14 (atomic density) values are generally greater than the z-profile density
# There is no reason that these values should align. The shape should be roughly the same though

# If we force the density profiles to use the same axis limits as the feature z-profiles, we can get away with one axis ticks

# Right now, we're just setting the distribution ylimits to be the max/min of the plotted features +-5%
# This won't align with the feature z-profiles because +-5% of the normalized range is not the same as +-5% of the raw range
# Further, we are currently ranging each feature distribution to +-5% of that feature's extrema
# This compared to the single range of the z-profiles will result in a mismatch

# Each feature distribution should be scaled to the a, b result given by the normalization

In [ ]:
alphabet = string.ascii_lowercase
feat_colors = ["darkgreen", "darkturquoise", "mediumvioletred"]
feat_labels = [
    rf"$\rho_{{{rad}}}$",
    r"$\Delta s$",
    r"$\mathit{U}$",
]  # replace with optim rad
labels = [r"$\rho$", r"$\rho_{l}$", r"$\rho_{v}$"]
plot_temps = [350, 375, 400]
nplot_temps = len(plot_temps)

####### bounds #######

train_mins = minmax[:, 0]
train_maxes = minmax[:, 1]

plot_mins = np.ones(nvars) * 1e9
plot_maxes = np.ones(nvars) * -1e9
plot_mask = np.isin(coexist_temps, plot_temps)

plot_features = np.array(features, object)[plot_mask]

for i, temp_feats in enumerate(plot_features):

    for j in range(nvars):

        mi = temp_feats[j].min()
        ma = temp_feats[j].max()

        if mi < plot_mins[j]:
            plot_mins[j] = mi
        if ma > plot_maxes[j]:
            plot_maxes[j] = ma

feat_ub_norm = (plot_maxes - train_mins) / (train_maxes - train_mins)
ub_norm = np.max(feat_ub_norm)

feat_lb_norm = (plot_mins - train_mins) / (train_maxes - train_mins)
lb_norm = np.min(feat_lb_norm)

span_norm = ub_norm - lb_norm
pad_norm = 0.05 * span_norm
a_norm = lb_norm - pad_norm
b_norm = ub_norm + pad_norm

a = a_norm * (train_maxes - train_mins) + train_mins
b = b_norm * (train_maxes - train_mins) + train_mins

plot_feat_z = np.array(feat_zs, object)[plot_mask]
z_min = 0
z_max = 1e9

for i in range(plot_feat_z.shape[0]):

    zbin, yy = plot_feat_z[i]
    zma = zbin.max()
    if zma < z_max:
        z_max = zma

##### end bounds #####

fig = plt.figure(figsize=(7.5, 5.5), dpi=400)
gs = fig.add_gridspec(
    nrows=nplot_temps,
    ncols=nvars + 1,
    height_ratios=[1] * nplot_temps,
    width_ratios=[2.5] + [1] * nvars,
    # wspace=0.25,
    # hspace=0.35,
)

left_axes = [fig.add_subplot(gs[i, 0]) for i in range(nplot_temps)]
right_axes = [
    [fig.add_subplot(gs[i, j + 1]) for j in range(3)] for i in range(nplot_temps)
]

bins = [50, 80, 40]  # number of bins for each feature dist.

for t, temp in enumerate(plot_temps):

    ti = np.where(coexist_temps == temp)[0][0]
    density_fit = lvc_densities[ti]  # fitted density z-profile
    kmeans_density = kmeans_densities[ti]  # kmeans density z-profile
    zbin, yy = feat_zs[ti]  # normalized feature z-profiles
    zbin_data, density_data = data_densities[ti]  # raw data density z-profile
    stride = 3  # plot every stride point for fit (circles)

    ax = left_axes[t]
    ax2 = ax.twinx()

    ##### Plot z-profiles #####
    ax.plot(
        density_fit[2],
        density_fit[1] / 1000,
        label=labels[0],
        color="black",
        zorder=0,
        linestyle="-",
    )
    ax.plot(
        kmeans_density[1],
        kmeans_density[0] / 1000,
        color="blue",
        zorder=1,
        linestyle="-",
        label=labels[1],
    )
    ax.plot(
        kmeans_density[4],
        kmeans_density[3] / 1000,
        color="firebrick",
        zorder=2,
        linestyle="-",
        label=labels[2],
    )

    ax.scatter(
        zbin_data[::stride],
        density_data[::stride] / 1000,
        marker="o",
        s=8,
        color="black",
        facecolors="white",
        linewidths=0.5,
    )
    ##### Plot feature distributions #####
    for i, y in enumerate(yy):

        ax2.plot(
            zbin,
            y,
            color=feat_colors[i],
            zorder=0 - i,
            linestyle="--",
            label=feat_labels[i],
        )

    ###### labels, limits #####
    ax.set_ylim(a_norm, b_norm)
    ax.set_xlim(z_min, z_max)
    ax.set_ylabel(r"$\rho \ (\mathrm{g/cm^{3}})$")
    ax.grid(alpha=0.5)
    # ax.grid(
    #     which="minor", linestyle=":", linewidth="0.5", color="lightgray", alpha=0.75
    # )

    ax2.set_ylim(a_norm, b_norm)
    ax2.set_yticklabels([])

    if t == nplot_temps - 1:
        ax.set_xlabel(r"$Z \ (\text{Å})$")
    else:
        ax.set_xticklabels([])

    ##### legend, annotations ####
    ax.text(
        9,
        0.97,
        rf"$\textbf{{{alphabet[t]}) {plot_temps[t]}\ K}}$",
        fontsize=10,
        weight="bold",
    )

    if t == 0:
        ax.legend(loc=(0.53, 0.51), handlelength=1)
        ax2.legend(loc=(0.73, 0.51), handlelength=1)

    locs = [(-0.4, 0.8), (-0.22, 0.8), (-0.49, 0.8)]  # feature dist. title locs
    labels = [
        rf"$\bm{{\rho}}_{{\textbf{{{rad}}}}} \ \textbf{{(g/cm}}^{{\textbf{{3}}}} \textbf{{)}}$",
        r"$\bm{\Delta s} \ \textbf{(Å)}$",
        r"$\bm{\mathit{U} \ \textbf{(kcal/mol)}$",
    ]  # feature dist title labels

    for i in range(nvars):

        ax = right_axes[t][i]
        # ax.yaxis.tick_right()

        # get and plot kmeans gas-liquid distributions
        distribution = kmeans.get_feature_distribution(
            features[ti][i], phases[ti], bins[i], all_counts=False
        )
        ax = kmeans.plot_feature_distribution(distribution, ax=ax)

        ax.set_ylim(a[i], b[i])

        # format negative signs for PE ticks
        if i == nvars - 1:
            ax.yaxis.set_major_formatter(FuncFormatter(plotting.shortminus_formatter))

        # Add feat distribution titles
        if t == 0:

            dummy = mlines.Line2D([], [], color="none", label=labels[i])
            ax.legend(handles=[dummy], loc=locs[i])

        # add liquid-vapor legend
        if t == 1 and i == 1:

            dummy_liq = mlines.Line2D([], [], color="navy", label="liquid")
            dummy_gas = mlines.Line2D([], [], color="darkred", label="vapor")
            ax.legend(handles=[dummy_liq, dummy_gas], loc="center", handlelength=1)

        # only set distribution xticks for last row
        if t < nplot_temps - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel("Probability")

        ax.grid(alpha=0.5)
        # ax.grid(
        #     which="minor", linestyle=":", linewidth="0.5", color="lightgray", alpha=0.75
        # )


fig.subplots_adjust(wspace=0.33, hspace=0.2)
fig.savefig(fig_path + f"demo_{paper_name}.png")
plt.show()

In [ ]:
print("Results dir:")
_ = [print(file) for file in sorted(os.listdir(results_dir))]
print("\n", "figs:", sep="")
_ = [print(file) for file in sorted(os.listdir(fig_path))]
print("\n", "data dir:", sep="")
_ = [print(file) for file in sorted(os.listdir(data_dir))]

In [ ]:
for file in sorted(os.listdir(data_dir)):
    if file.endswith("_b.dump") or file.endswith(".DS_Store"):
        os.remove(data_dir + file)